In [22]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as st
import re
import os

In [23]:
def bins(flt1, flt2):
    if flt1 == 0 and flt2 >= 2:
        return ("loss_retarget")
    if flt1 == 0 and flt2 == 1:
        return ("loss_noretarget")
    if flt1 >= 1 and flt2 == 1:
        return("present_noretarget")
    if flt1 >= 1 and flt2 >= 2:
        return("present_retarget")
    if flt1 >= 1 and flt2 == 0:
        return("present_unknowntargeting")
    if flt1 ==0  and flt2 == 0:
        return("loss_unknowntargeting")
    else:
        print("input is not float between 0 and inf")      

def binscol(df):
    output= pd.DataFrame(columns=species)
    for spp in species:
        predictions=[]
        column1 = "".join([spp," Presence/Absence"])
        column2 = "".join([spp," Retargeting"])
        for i in range(0,len(df)):
            flt1 = df[column1][i]
            flt2 = df[column2][i]
            prediction = bins(flt1,flt2)
            predictions.append(prediction)
        output[spp] = predictions
        #output['description'] = df.index
        output = output.set_index(df.index)
    return(output)

def fisher(df,factor,cols):
    dfcounts=df.loc[:,cols].groupby(factor).sum()
    oddsratio, pvalue = st.fisher_exact([[dfcounts.iloc[0,0], dfcounts.iloc[0,1]],[dfcounts.iloc[1,0],dfcounts.iloc[1,1]]])
    print(dfcounts,"\n","The Fisher's pvalue is", pvalue)


In [24]:
#import absence-presence matrix
APmat = pd.read_csv("20240805_aaRS_processing_collapsemodels.csv", index_col=["targeting", "description"]).rename(columns ={
    'Arabidopsis thaliana' : 'A. thaliana',
    'Balanophera fungosa' : 'B. fungosa',
    "Phalaenopsis equestris":'P. equestris' ,
    "Epipogium aphyllum":'E. aphyllum',
    "Epipogium roseum":'E. roseum',	
    "Manihot esculenta":"M. esculenta",	
    "Rafflesia cantleyi": "R. cantleyi",
    "Sapria himalayana" :'S. himalayana',	
    "Santalum album":'S. album',
    "Rhopalocnemis phalloides":'R. phalloides',
    "Balanophora fungosa":'B. fungosa',
    "Rhododendron williamsianum":'R. williamsianum',	
    'Monotropa hypopitys':'M. hypopitys'})
APmat.columns


Index(['A. thaliana', 'Amborella trichopoda', 'B. fungosa',
       'Cuscuta australis', 'Cuscuta campestris', 'Dendropemon caribaeus',
       'E. aphyllum', 'E. roseum', 'M. hypopitys', 'Ipomoea trifida',
       'M. esculenta', 'Mimulus guttatus', 'Malania oleifera', 'Oryza sativa',
       'P. equestris', 'Phtheirospermum japonicum', 'R. cantleyi',
       'R. phalloides', 'R. williamsianum', 'S. album', 'Striga asiatica',
       'S. himalayana', 'Spirodela polyrhiza', 'Viscum album',
       'Vitis vinifera'],
      dtype='object')

In [25]:
#import predicted retargeting matrix for mito and chloro
MTmat = pd.read_csv("20240819_retargetingmatrix_mito.csv",index_col=["targeting","description"])

#just select the cytosolic aaRS
MTmat=MTmat.loc["cytosolic",:]
MTmat.head(20)

,P. equestris,E. aphyllum,E. roseum,M. esculenta,R. cantleyi,S. himalayana,S. album,R. phalloides,B. fungosa,R. williamsianum,M. hypopitys
description,,,,,,,,,,,
AlaRS,1.0,1.0,1.0,3.0,1.0,3.0,3.0,1.0,1.0,1.0,3.0
AsnRS,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0
AspRS,1.0,1.0,1.0,1.0,1.0,3.0,1.0,1.0,3.0,0.0,1.0
CysRS,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
GlnRS,1.0,1.0,1.0,1.0,0.0,2.0,1.0,1.0,1.0,1.0,1.0
GluRS,1.0,1.0,1.0,1.0,0.0,2.0,1.0,0.0,0.0,1.0,0.0
GlyRS,2.0,3.0,1.0,3.0,1.0,3.0,1.0,1.0,1.0,1.0,0.0
HisRS,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
IleRS,1.0,1.0,3.0,1.0,2.0,1.0,1.0,1.0,2.0,1.0,1.0


In [26]:
CTmat = pd.read_csv("20240819_retargetingmatrix_chloro.csv",index_col=["targeting","description"])
#just select the cytosolic aaRS
CTmat=CTmat.loc["cytosolic",:]
CTmat.head(20)


,P. equestris,E. aphyllum,E. roseum,M. esculenta,R. cantleyi,S. himalayana,S. album,R. phalloides,B. fungosa,R. williamsianum,M. hypopitys
description,,,,,,,,,,,
AlaRS,3.0,3.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,3.0,1.0
AsnRS,1.0,1.0,1.0,1.0,0.0,1.0,1.0,2.0,1.0,1.0,1.0
AspRS,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,0.0,1.0
CysRS,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
GlnRS,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0
GluRS,1.0,1.0,1.0,1.0,0.0,3.0,1.0,0.0,0.0,1.0,0.0
GlyRS,2.0,1.0,2.0,1.0,1.0,2.0,1.0,2.0,1.0,1.0,0.0
HisRS,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
IleRS,1.0,1.0,1.0,1.0,3.0,1.0,1.0,1.0,2.0,1.0,1.0


In [27]:
#convert R cat and S him values to 0 except for GluRS
#overwritespp = ['R. cantleyi','S. himalayana']
# CTmat[overwritespp]=pd.concat([CTmat.loc[CTmat.index.get_level_values('description')!= 'GluRS',overwritespp].replace([1,2,3],0),
# CTmat.loc[CTmat.index.get_level_values('description')== 'GluRS',overwritespp]])
CTmat

,P. equestris,E. aphyllum,E. roseum,M. esculenta,R. cantleyi,S. himalayana,S. album,R. phalloides,B. fungosa,R. williamsianum,M. hypopitys
description,,,,,,,,,,,
AlaRS,3.0,3.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,3.0,1.0
AsnRS,1.0,1.0,1.0,1.0,0.0,1.0,1.0,2.0,1.0,1.0,1.0
AspRS,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,0.0,1.0
CysRS,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
GlnRS,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0
GluRS,1.0,1.0,1.0,1.0,0.0,3.0,1.0,0.0,0.0,1.0,0.0
GlyRS,2.0,1.0,2.0,1.0,1.0,2.0,1.0,2.0,1.0,1.0,0.0
HisRS,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
IleRS,1.0,1.0,1.0,1.0,3.0,1.0,1.0,1.0,2.0,1.0,1.0


In [28]:
MTmat= MTmat.drop(labels='PheRSB', axis=0).rename(index={'PheRSA':'PheRS'}) #drop PheRS b as we are only going to count as retargeted if both subunits are
CTmat= CTmat.drop(labels='PheRSB', axis=0).rename(index={'PheRSA':'PheRS'}) #drop PheRS b as we are only going to count as retargeted if both subunits are

In [29]:
species= list(MTmat.columns)
species

['P. equestris',
 'E. aphyllum',
 'E. roseum',
 'M. esculenta',
 'R. cantleyi',
 'S. himalayana',
 'S. album',
 'R. phalloides',
 'B. fungosa',
 'R. williamsianum',
 'M. hypopitys']

In [30]:
#Add Max of all GatCAB as GLNRS
GlnRS = APmat.filter(regex='Gat', axis=0).max(axis=0)
APmat= APmat.filter(regex='RS', axis=0)
APmatorg = APmat.loc['organellar', species]
APmatorg.loc["GlnRS"] = GlnRS
APmatorg = APmatorg.sort_index()
APmatorg

,P. equestris,E. aphyllum,E. roseum,M. esculenta,R. cantleyi,S. himalayana,S. album,R. phalloides,B. fungosa,R. williamsianum,M. hypopitys
description,,,,,,,,,,,
AlaRS,2,0,0,2,0,0,2,0,0,2,0
AsnRS,1,1,1,2,0,0,2,0,2,2,2
AspRS,2,2,2,2,0,0,2,1,0,2,2
CysRS,2,1,1,2,0,0,2,0,0,2,0
GlnRS,2,2,2,2,0,0,2,1,0,2,2
GluRS,2,2,1,2,0,0,2,2,2,2,2
GlyRS,2,0,0,2,0,0,2,0,0,2,2
HisRS,2,2,1,2,0,0,2,0,0,2,2
IleRS,2,2,1,2,0,0,2,2,0,2,2


In [31]:
mergedchloro = pd.merge(APmatorg,CTmat, how='inner', on='description') 
mergedchloro.columns= [c.replace("_x"," Presence/Absence") for c in list(mergedchloro.columns)]
mergedchloro.columns= [c.replace("_y"," Retargeting") for c in list(mergedchloro.columns)]
binschloro = binscol(mergedchloro)
binschloro["Genome"] = "plastid"
binschloro.to_csv("20240820_absencepresence_retarget_chloro.csv")
binschloro

/tmp/ipykernel_7372/2274605934.py:24: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  flt1 = df[column1][i]
/tmp/ipykernel_7372/2274605934.py:25: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  flt2 = df[column2][i]


,P. equestris,E. aphyllum,E. roseum,M. esculenta,R. cantleyi,S. himalayana,S. album,R. phalloides,B. fungosa,R. williamsianum,M. hypopitys,Genome
description,,,,,,,,,,,,
AlaRS,present_retarget,loss_retarget,loss_noretarget,present_noretarget,loss_noretarget,loss_noretarget,present_noretarget,loss_retarget,loss_retarget,present_retarget,loss_noretarget,plastid
AsnRS,present_noretarget,present_noretarget,present_noretarget,present_noretarget,loss_unknowntargeting,loss_noretarget,present_noretarget,loss_retarget,present_noretarget,present_noretarget,present_noretarget,plastid
AspRS,present_noretarget,present_noretarget,present_noretarget,present_noretarget,loss_noretarget,loss_retarget,present_noretarget,present_noretarget,loss_noretarget,present_unknowntargeting,present_noretarget,plastid
CysRS,present_noretarget,present_noretarget,present_noretarget,present_noretarget,loss_noretarget,loss_noretarget,present_noretarget,loss_noretarget,loss_noretarget,present_noretarget,loss_noretarget,plastid
GlnRS,present_noretarget,present_noretarget,present_noretarget,present_noretarget,loss_unknowntargeting,loss_noretarget,present_noretarget,present_noretarget,loss_noretarget,present_noretarget,present_noretarget,plastid
GluRS,present_noretarget,present_noretarget,present_noretarget,present_noretarget,loss_unknowntargeting,loss_retarget,present_noretarget,present_unknowntargeting,present_unknowntargeting,present_noretarget,present_unknowntargeting,plastid
GlyRS,present_retarget,loss_noretarget,loss_retarget,present_noretarget,loss_noretarget,loss_retarget,present_noretarget,loss_retarget,loss_noretarget,present_noretarget,present_unknowntargeting,plastid
HisRS,present_noretarget,present_noretarget,present_noretarget,present_noretarget,loss_noretarget,loss_noretarget,present_noretarget,loss_unknowntargeting,loss_noretarget,present_noretarget,present_noretarget,plastid
IleRS,present_noretarget,present_noretarget,present_noretarget,present_noretarget,loss_retarget,loss_noretarget,present_noretarget,present_noretarget,loss_retarget,present_noretarget,present_noretarget,plastid


In [32]:
mergedmito = pd.merge(APmatorg,MTmat, how='inner', on='description') 
mergedmito.columns= [c.replace("_x"," Presence/Absence") for c in list(mergedmito.columns)]
mergedmito.columns= [c.replace("_y"," Retargeting") for c in list(mergedmito.columns)]
binsmito = binscol(mergedmito)
binsmito["Genome"] = "mitochondrion"
binsmito.to_csv("20240820_absencepresence_retarget_mito.csv")
binsmito

/tmp/ipykernel_7372/2274605934.py:24: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  flt1 = df[column1][i]
/tmp/ipykernel_7372/2274605934.py:25: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  flt2 = df[column2][i]


,P. equestris,E. aphyllum,E. roseum,M. esculenta,R. cantleyi,S. himalayana,S. album,R. phalloides,B. fungosa,R. williamsianum,M. hypopitys,Genome
description,,,,,,,,,,,,
AlaRS,present_noretarget,loss_noretarget,loss_noretarget,present_retarget,loss_noretarget,loss_retarget,present_retarget,loss_noretarget,loss_noretarget,present_noretarget,loss_retarget,mitochondrion
AsnRS,present_noretarget,present_noretarget,present_noretarget,present_noretarget,loss_unknowntargeting,loss_noretarget,present_noretarget,loss_noretarget,present_noretarget,present_noretarget,present_noretarget,mitochondrion
AspRS,present_noretarget,present_noretarget,present_noretarget,present_noretarget,loss_noretarget,loss_retarget,present_noretarget,present_noretarget,loss_retarget,present_unknowntargeting,present_noretarget,mitochondrion
CysRS,present_noretarget,present_noretarget,present_noretarget,present_noretarget,loss_noretarget,loss_noretarget,present_noretarget,loss_noretarget,loss_noretarget,present_noretarget,loss_noretarget,mitochondrion
GlnRS,present_noretarget,present_noretarget,present_noretarget,present_noretarget,loss_unknowntargeting,loss_retarget,present_noretarget,present_noretarget,loss_noretarget,present_noretarget,present_noretarget,mitochondrion
GluRS,present_noretarget,present_noretarget,present_noretarget,present_noretarget,loss_unknowntargeting,loss_retarget,present_noretarget,present_unknowntargeting,present_unknowntargeting,present_noretarget,present_unknowntargeting,mitochondrion
GlyRS,present_retarget,loss_retarget,loss_noretarget,present_retarget,loss_noretarget,loss_retarget,present_noretarget,loss_noretarget,loss_noretarget,present_noretarget,present_unknowntargeting,mitochondrion
HisRS,present_noretarget,present_noretarget,present_noretarget,present_noretarget,loss_noretarget,loss_noretarget,present_noretarget,loss_unknowntargeting,loss_noretarget,present_noretarget,present_noretarget,mitochondrion
IleRS,present_noretarget,present_noretarget,present_retarget,present_noretarget,loss_retarget,loss_noretarget,present_noretarget,present_noretarget,loss_retarget,present_noretarget,present_noretarget,mitochondrion


In [34]:
both=pd.concat([binsmito,binschloro])
both.to_csv("20240820_absencepresence_retarget.csv")